# 15 Keyframes: FLUX.2 klein 4B on a Kaggle T4
Used when Cloudflare's daily image quota is spent. Settings: GPU T4 x2, Internet on, Persistence Files only. Add the dataset that holds `image_jobs.json` (from `python -m pipeline.kaggle_images`). It saves each image as it goes and skips finished ones, so after a timeout just run it again.

In [ ]:
!pip install -q -U diffusers transformers accelerate

In [ ]:
%%writefile /kaggle/working/gen_keyframes.py
import json, shutil, time
from pathlib import Path
import torch
from PIL import Image
from diffusers import Flux2KleinPipeline

JOBS = next(Path("/kaggle/input").rglob("image_jobs.json"))  # from `python -m pipeline.kaggle_images`
OUT = Path("/kaggle/working/images")
LOG = Path("/kaggle/working/notebook_log.jsonl")
jobs = json.loads(JOBS.read_text())
print(len(jobs), "jobs,", sum((OUT / j["out"]).exists() for j in jobs), "already done")

MODEL = "black-forest-labs/FLUX.2-klein-4B"  # Apache 2.0, not gated
try:  # two T4s: spread the text encoder and the transformer over both GPUs
    pipe = Flux2KleinPipeline.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="balanced")
    placement = "balanced"
except Exception as e:
    print("balanced placement failed, using CPU offload:", e)
    pipe = Flux2KleinPipeline.from_pretrained(MODEL, torch_dtype=torch.bfloat16)
    pipe.enable_model_cpu_offload()
    placement = "cpu_offload"
try:  # the VAE decode ran out of memory on a T4 at full size (FAILURES #12): decode in tiles
    pipe.vae.enable_tiling()
except Exception as e:
    print("no vae tiling:", e)

for job in jobs:
    torch.cuda.empty_cache()
    out = OUT / job["out"]
    if out.exists():
        continue
    kwargs = dict(prompt=job["prompt"], width=job["width"], height=job["height"], guidance_scale=1.0,
                  num_inference_steps=4, generator=torch.Generator("cuda").manual_seed(job["seed"]))
    if job["refs"]:
        kwargs["image"] = [Image.open(OUT / r).convert("RGB") for r in job["refs"]]
    start = time.time()
    image = pipe(**kwargs).images[0]
    seconds = time.time() - start
    out.parent.mkdir(parents=True, exist_ok=True)
    image.save(out)
    with LOG.open("a") as f:
        f.write(json.dumps({"stage": "keyframes", "model": MODEL, "out": job["out"], "seed": job["seed"],
                            "seconds": round(seconds, 1), "placement": placement}) + "\n")
    print(job["out"], round(seconds, 1), "s")

shutil.make_archive("/kaggle/working/images_out", "zip", OUT)
print("done: download /kaggle/working/images_out.zip and notebook_log.jsonl")

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python /kaggle/working/gen_keyframes.py